# [Chapter 18 — Dengue, §18.6] Vector-Control Intervention Sensitivity Analysis

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 18 — Dengue
**Considerations developed:** 11 (correct sensitivity), 12 (basic vs effective $R$)
**Estimated runtime:** ≤ 20 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_18_dengue/02_vector_intervention_sensitivity.ipynb)


## What this notebook does

Computes normalized sensitivity indices of the Ross-Macdonald $\mathcal{R}_0$ with respect to four intervention-relevant parameters: vector population reduction (insecticide / source reduction), biting-rate reduction (bed nets), vector longevity reduction (larvicides), and human-recovery acceleration (faster diagnosis + supportive care). Identifies the highest-leverage intervention for each baseline parameter set.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import csv, json
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_18
from shared.verification import assert_within_tolerance
set_seed_chapter_18()
book_style()

# Path to the synthetic-data root for the case studies
DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'data'))


## Sensitivity analysis on Ross-Macdonald R_0

$$\mathcal{R}_0 = \sqrt{\frac{a^2 b c (M/N)}{\mu_v \gamma_h}}$$
Closed-form normalized sensitivity indices:
$$S^{R_0}_a = +1,\quad S^{R_0}_{M/N} = +0.5,\quad S^{R_0}_b = +0.5,\quad S^{R_0}_{\mu_v} = -0.5,\quad S^{R_0}_{\gamma_h} = -0.5$$

In [ ]:
from shared.parameters import baseline_chapter_18
import math
p = baseline_chapter_18()
MN = (p['M_over_N_dry'] + p['M_over_N_wet'])/2

params = {
    'a':     {'value': p['a'],            'sensitivity': +1.0,   'reduction_via': 'bed nets, repellent'},
    'M_over_N': {'value': MN,             'sensitivity': +0.5,   'reduction_via': 'insecticide spraying, source reduction'},
    'b':     {'value': p['b'],            'sensitivity': +0.5,   'reduction_via': 'protective clothing, repellent'},
    'mu_v':  {'value': p['mu_v_baseline'], 'sensitivity': -0.5,   'reduction_via': 'larvicides, biological control'},
    'gamma_h': {'value': p['gamma_h'],    'sensitivity': -0.5,   'reduction_via': 'faster diagnosis + care'},
}

print(f'{"parameter":<10}  {"|S|":>6}  {"intervention":<40}')
print('-'*68)
for k, v in params.items():
    print(f'{k:<10}  {abs(v["sensitivity"]):>6.2f}  {v["reduction_via"]}')
print('\nHighest-leverage parameter: a (biting rate) -- bed nets have 2x impact of vector reduction')


## Numerical verification of the closed-form sensitivities

In [ ]:
def R0(a_v, b_v, c_v, MN_v, mu_v_v, gamma_v):
    return math.sqrt(a_v**2 * b_v * c_v * MN_v / (mu_v_v * gamma_v))

# Baseline R_0
R0_base = R0(p['a'], p['b'], p['c'], MN, p['mu_v_baseline'], p['gamma_h'])
print(f'Baseline R_0 = {R0_base:.3f}')

h = 1e-4
fd = {}
for k, v in params.items():
    val = v['value']
    if k == 'a':       up = R0(val*(1+h), p['b'], p['c'], MN, p['mu_v_baseline'], p['gamma_h'])
    elif k == 'b':     up = R0(p['a'], val*(1+h), p['c'], MN, p['mu_v_baseline'], p['gamma_h'])
    elif k == 'M_over_N': up = R0(p['a'], p['b'], p['c'], val*(1+h), p['mu_v_baseline'], p['gamma_h'])
    elif k == 'mu_v':  up = R0(p['a'], p['b'], p['c'], MN, val*(1+h), p['gamma_h'])
    elif k == 'gamma_h': up = R0(p['a'], p['b'], p['c'], MN, p['mu_v_baseline'], val*(1+h))
    sens = (up - R0_base) / R0_base / h
    fd[k] = sens
    expected = v['sensitivity']
    print(f'{k:<10}  numeric S={sens:+.3f}  closed-form S={expected:+.2f}  match={abs(sens-expected)<0.01}')

for k in params:
    assert abs(fd[k] - params[k]['sensitivity']) < 0.01, f'sensitivity mismatch for {k}'
print('\nVerified: numerical sensitivities match closed-form expressions.')


## Tornado plot

In [ ]:
import matplotlib.pyplot as plt
names = list(params.keys())
sens = [params[k]['sensitivity'] for k in names]
order = np.argsort(np.abs(sens))[::-1]
names_s = [names[i] for i in order]
sens_s = [sens[i] for i in order]
colors = [BOOK_COLORS['primary'] if s > 0 else BOOK_COLORS['secondary'] for s in sens_s]

fig, ax = plt.subplots(figsize=(8, 4))
y = np.arange(len(names_s))
ax.barh(y, sens_s, color=colors)
ax.set_yticks(y)
ax.set_yticklabels(names_s)
ax.set_xlabel('Normalized sensitivity index $S^{R_0}_p$')
ax.set_title('Tornado plot: which parameter to target for R_0 reduction')
ax.axvline(0, color='black', lw=0.5)
ax.grid(True, alpha=0.3, axis='x')
fig.tight_layout()
plt.show()


## What's next

- Chapter 10's tornado-plot machinery generalizes this to arbitrary models without closed forms.
- The bed-net 2x advantage is a robust qualitative finding but the exact ranking depends on local implementation costs and effective coverage rates.
- The next case study (Chapter 19 HIV) replaces vector-control levers with treatment-as-prevention and shows how the same sensitivity framework adapts to different intervention classes.